In [ ]:
from sampo.generator.base import SimpleSynthetic
from sampo.generator.environment.contractor_by_wg import get_contractor_by_wg as get_contractor
from sampo.scheduler.genetic.base import GeneticScheduler
from sampo.scheduler.genetic.operators import TimeAndResourcesFitness
from sampo.utilities.resource_usage import resources_peaks_sum, resources_costs_sum, resources_sum
import pandas as pd

## Set parameters and generate synthetic graph

In [ ]:
def get_graph(graph_size, seed):
    return SimpleSynthetic(seed).work_graph(bottom_border=graph_size)

def get_graph_size_diff(graph_size, seed):
    graph = get_graph(graph_size, seed)
    real_size = len(graph)
    return real_size - graph_size

def get_optimal_graph(graph_size, search_seeds=1000):
    optimal_seed = min(
        range(search_seeds), 
        key=lambda seed: abs(get_graph_size_diff(graph_size, seed))
    )
    optimal_graph = get_graph(graph_size, optimal_seed)
    return optimal_graph

In [ ]:
GRAPH_SIZE = 52
SEED = 1

POPULATION_SIZE = 200
N_GENERATIONS = 100

MUTATION_P = 0.01

fitness_constructor = TimeAndResourcesFitness()
fitness_weights = (-1, -1)
is_multiobjective = True
optimize_resources = True

In [ ]:
graph = get_optimal_graph(GRAPH_SIZE, search_seeds=1000)
contractors = get_contractor(graph),
print(len(graph))

## Use the genetic algorithm and save history

In [ ]:
from itertools import product
from random import Random
rng = Random(SEED)

In [ ]:
# (mating_type, mutation_type)
# mating_type = (pairing_type, order_crossover, resources_crossover)
# mutation_type = (mutation_type, n_mutations)
2*9*2

In [ ]:
all_types = [
    # (("RANDOM", "TWO_POINT", "BY_WORK"), ("CLASSIC", 1)),
    
    (("RANDOM", "ONE_POINT", "SHUFFLE"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "BY_WORK"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "WEIGHTED_AND_SHUFFLE"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "WEIGHTED"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "BY_ORDER"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "MIN_MAX"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "MIN_MAX_BY_WORK"), ("CLASSIC", 1)),
    (("RANDOM", "ONE_POINT", "MIN_MAX_BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "SKIP", "SKIP"), ("CLASSIC", 1)),

    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "SHUFFLE"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "BY_WORK"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "BY_WORKER"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "WEIGHTED_AND_SHUFFLE"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "WEIGHTED"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "BY_ORDER"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "MIN_MAX"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "MIN_MAX_BY_WORK"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "ONE_POINT", "MIN_MAX_BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "SKIP", "SKIP"), ("CLASSIC", 1)),

    (("RANDOM", "TWO_POINT", "SHUFFLE"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "BY_WORK"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "WEIGHTED_AND_SHUFFLE"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "WEIGHTED"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "BY_ORDER"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "MIN_MAX"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "MIN_MAX_BY_WORK"), ("CLASSIC", 1)),
    (("RANDOM", "TWO_POINT", "MIN_MAX_BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "SKIP", "SKIP"), ("CLASSIC", 1)),

    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "SHUFFLE"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "BY_WORK"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "BY_WORKER"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "WEIGHTED_AND_SHUFFLE"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "WEIGHTED"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "BY_ORDER"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "MIN_MAX"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "MIN_MAX_BY_WORK"), ("CLASSIC", 1)),
    (("PHENOTYPE_CLUSTERS", "TWO_POINT", "MIN_MAX_BY_WORKER"), ("CLASSIC", 1)),
    (("RANDOM", "SKIP", "SKIP"), ("CLASSIC", 1)),

]
len(all_types)

In [ ]:
n_experiment_repeats = 5
for experiment_id in range(n_experiment_repeats):
        
    new_seed = SEED + experiment_id

    rng = Random(new_seed)
    settings_for_each_generation = []
    for n_mutations in range(100):
        shuffled_types = rng.sample(all_types, len(all_types))
        settings_for_each_generation.extend(shuffled_types)
    
    genetic_algorithm = GeneticScheduler(
        number_of_generation=N_GENERATIONS,
        size_of_population=POPULATION_SIZE,
        
        mutate_order=MUTATION_P,
        mutate_resources=MUTATION_P,
        
        fitness_constructor=fitness_constructor,
        fitness_weights=fitness_weights,
        is_multiobjective=is_multiobjective,
        optimize_resources=optimize_resources,
        
        settings_for_each_generation=settings_for_each_generation,
        seed=new_seed,
        save_history_to=f"history/data250/ONE_TWO_POINT_{experiment_id}.json"
    )
    
    genetic_result = genetic_algorithm.schedule(graph, contractors)

## Plot Pareto-fronts

In [ ]:
from os import listdir
import numpy as np
import pandas as pd
import json
import plotly.express as px

from sampo.scheduler.utils.fitness_history import FitnessHistorySummary

In [ ]:
df = pd.concat([
    pd.DataFrame(FitnessHistorySummary.load_json(f"history/data250/{file}").pareto_front_history[-1]).assign(experiment=file)
    for file in listdir("history/data100")
])

In [ ]:
fig = px.line(
    df, x=0, y=1, color="experiment",
    template="plotly_white"
)
fig.update_traces(mode="lines+markers")

experiment_to_color = {
    "GA_": "black",
    "ONE_TWO_POINT_": "green"
}
for i, trace in enumerate(fig.data):
    for experiment, color in experiment_to_color.items():
        if trace.name.startswith(experiment):
            fig.data[i].line.color=color

fig.update_layout(height=1000, width=1000, showlegend=True)
# fig.write_image("pareto.png", scale=3)
fig.show()

In [ ]:
df.groupby("experiment").mean()

## Get summary of the evolution

In [ ]:
from sampo.scheduler.utils.fitness_history import FitnessHistorySummary

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"

In [ ]:
# Load history from file
summary = FitnessHistorySummary.load_json("history/data250/ONE_TWO_POINT_0.json")

In [ ]:
population_means = np.array(summary.get_fitness_means())

fig = px.line(y=population_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
population_means = np.array(summary.get_fitness_means())

fig = px.line(y=population_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_front_means = np.array(summary.get_fitness_means(only_pareto=True))

fig = px.line(y=pareto_front_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_front_means = np.array(summary.get_fitness_means(only_pareto=True))

fig = px.line(y=pareto_front_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
offsprings_means = np.array(summary.get_fitness_means(only_offsprings=True))

fig = px.line(y=offsprings_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
offsprings_means = np.array(summary.get_fitness_means(only_offsprings=True))

fig = px.line(y=offsprings_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_ratios = summary.get_pareto_to_population_ratios()

fig = px.line(y=pareto_ratios)
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
population_shifts = summary.get_generation_shifts()
pareto_front_shifts = summary.get_generation_shifts(only_pareto=True)

fig = px.line(y=[population_shifts, pareto_front_shifts])
fig.data[1].line.color = "white"
fig.update_layout(height=500, width=1000, showlegend=False)
fig.show()

In [ ]:
population_uniqueness = summary.get_uniqueness_scores()
pareto_uniqueness = summary.get_uniqueness_scores(only_pareto=True)

fig = px.line(y=[population_uniqueness, pareto_uniqueness])
fig.data[1].line.color = "white"
fig.update_layout(height=500, width=1000, showlegend=False)
fig.show()

In [ ]:
from os import listdir

In [ ]:
data = []
# folder = 'history/data150'
for folder in ('history/data100', 'history/data150', 'history/data200', 'history/data250'):
    for file in listdir(folder):
        # if not (file.startswith('GA_') or file.startswith('ONE_')):
        #     continue
        
        summary = FitnessHistorySummary.load_json(f"{folder}/{file}")
        population_shifts = summary.get_generation_shifts()
        df = pd.DataFrame(dict(
            comments=[tuple(tuple(j) for j in i) for i in summary.comments[1:]],
            population_shifts=population_shifts
        ))
        data.append(df)
        
    
d = pd.concat(data)

In [ ]:
d.groupby('comments').mean().squeeze().sort_values(ascending=False)

In [ ]:
d.groupby(d['comments'].apply(lambda x: x[0][1])).mean(numeric_only=True).squeeze().sort_values(ascending=False)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
df = pd.DataFrame(summary.population_history[-25])
clusters = KMeans(n_clusters=8).fit_predict(StandardScaler().fit_transform(df))

In [ ]:
fig = px.scatter(
    df.assign(clusters=clusters),
    x=0, y=1,
    color='clusters',
    template='plotly_white',
    color_continuous_scale=['black', '#008C45', '#008C45', 'black', '#008C45', 'black', 'black', '#008C45']
)
fig.update_layout(coloraxis_showscale=False, plot_bgcolor='#fcfcfc', paper_bgcolor='#fcfcfc')
fig.update_layout(height=1080//2, width=1080//2, margin_t=0, margin_b=0, margin_l=0, margin_r=0)
fig.update_xaxes(title_text="")
fig.update_yaxes(title_text="")
# fig.write_image("plots/clusters.png", scale=4)
fig.show()